In [7]:
from USGSreportmaker import ReportMaker
from datetime import datetime, UTC
from zoneinfo import ZoneInfo
from manage_reports import init_reports_table, store_report_msg, select_report_msgs
from embeds import make_mmi_embed
import discord

In [ ]:
for msg_data in select_report_msgs('ci41292687'):
    guild_id, channel_id, msg_id = msg_data
    print(f'{guild_id = }, {channel_id = }, {msg_id = }')

In [2]:
rm = ReportMaker()

Fetched USGS data for query.


In [23]:
rm.load_ev_detail(19)

Loading report:
M 5.6 - 11 km N of Redwood Valley, CA
Jun 24, 2026 08:10 AM


Loading report:
M 4.3 - 18 km WSW of Johannesburg, CA
Jul 13, 2026 09:40 AM
EEW report loaded


In [ ]:
# tracemalloc.start()

index = 1
rm_temp = ReportMaker() # new insance to avoid editing the auto-reports

rm_temp.load_ev_detail(index,is_temp=True)

# check if event happened before launch of ShakeAlert
tformat = "%b %d, %Y %I:%M %p"
SAlaunch = "Oct 1, 2019 12:00 AM"
before_SA = datetime.strptime(rm_temp.ev_timestamp,tformat) < datetime.strptime(SAlaunch,tformat)

if rm_temp.has_eew:
    rm_temp.make_eew_map(is_temp=True)
    msg1 = (
        f"This earthquake triggered ShakeAlert.\n"
        f"An alert was sent to the following regions/counties:\n"
        f"-{"\n-".join(rm_temp.formatted_warned_areas)}\n"
    )
    print(msg1)
elif before_SA:
    print("This earthquake occurred before the launch of ShakeAlert.")
else:
    print("This earthquake did not trigger ShakeAlert.")

rm_temp.make_mmi_map(is_temp=True)

if rm_temp.mmi_plottable:
    msg2 = (
        f"On {rm_temp.ev_timestamp}\n"
        f"{rm_temp.mmi_report_caption}\n"
        f"Magnitude: {rm_temp.ev_mag}\n"
        f"Maximum intensity: {rm_temp.ev_maxnumeral} ({rm_temp.ev_maxdesc})\n"
        f"Maximum intensity felt in the following cities:\n"
        f"-{"\n-".join(rm_temp.cities_max_mmi)}\n\n"
    )
    print(msg2)
else:
    msg2_alt = (
        f"On {rm_temp.ev_timestamp}\n"
        f"A magnitude {rm_temp.ev_mag} occurred in the region.\n"
        f"No intensity-by-city information is available to plot for this earthquake.\n"
        f"For more details visit {rm_temp.ev_url}"
    )
    print(msg2_alt)

# current, peak = tracemalloc.get_traced_memory()
# print(f"Current memory usage: {current / 10**6} MB")
# print(f"Peak memory usage: {peak / 10**6} MB")

# tracemalloc.stop()

In [ ]:
rm.load_ev_detail(1600,is_temp=True)
rm.get_eew_data()
rm.make_eew_map()

In [ ]:
rm.get_mmi_data()
rm.make_mmi_map()

In [ ]:
with open("latest_report.txt") as f:
    lines = f.readlines()
    curr_ev_id = lines[0].strip()
    curr_ev_lastupdate = lines[1].strip()

rm = ReportMaker()
index = 0
rm.load_ev_detail(index=index)
# print(rm.evlist)

ev_updated = False
# if last bot event is same as API last event
if curr_ev_id == rm.ev_id:
    print("No new event.")
    # check for update on same event
    if int(curr_ev_lastupdate) == rm.ev_lastupdate:
        # do nothing if no updates
        print("No updates on latest event.")
    else:
        # remake maps for update
        # no need for new EEW map, only MMI
        print("Latest event updated.")
        ev_updated = True
        rm.make_mmi_map()
        mmi_header = "The latest earthquake report by the USGS has been updated."
else:
    # new event
    # make both maps
    print("New event posted.")
    rm.make_eew_map()
    rm.make_mmi_map()
    eew_header = "A new ShakeAlert product has been published by the USGS."
    mmi_header = "A new earthquake report has been published by the USGS."

if index != 0:
    try:
        eew_header = eew_header + "\n\n**THIS IS A TEST**"
    except: pass
    mmi_header = mmi_header + "\n\n**THIS IS A TEST**"


# do this for each valid print
if rm.has_eew:
    alert_message = (f"_{eew_header}_\n\n"
            f"A recent earthquake has triggered the ShakeAlert system.\n"
            f"An alert was sent to the following regions/counties:\n"
            f"- {"\n- ".join(rm.formatted_warned_areas)}\n"
            f"If you receive an earthquake alert\n"
            f"**drop, cover, and hold on.**"
            )
    print(alert_message)

report_message =  (f"_{mmi_header}_\n\n"
        # f"_Message generated {datetime.now()}_"
        f"**{rm.ev_timestamp}**\n"
        f"**{rm.mmi_report_caption}**\n"
        f"Magnitude: {rm.ev_mag}\n"
        f"Maximum intensity: {rm.ev_maxnumeral} ({rm.ev_maxdesc})\n"
        f"Maximum intensity felt in the following cities:\n"
        f"- {"\n- ".join(rm.cities_max_mmi)}\n\n"
        f"If you felt this earthquake, visit {rm.ev_url+"/tellus"}"
        f" to fill out a Did You Feel It report.\n\n\n"
    )

if rm.mmi_plottable:

    # if event update edit report message
    if ev_updated:
        # update time to datetime
        update_dt = datetime.fromtimestamp(rm.ev_lastupdate/1000, UTC)
        # to pacific time
        update_pt = update_dt.astimezone(ZoneInfo("America/Los_Angeles"))
        #to time string
        update_str = update_pt.strftime("%b %d, %Y %I:%M %p")
        try:
            print(report_message+f"_This event was last updated on {update_str}_")
        except:
            print("No ID available")
    #if new event send full new report
    else:  
        latest_mmi_msg = print(report_message)
else:
    report_message = (f"_{mmi_header}_\n\n"
        f"On {rm.ev_timestamp}\n"
        f"A magnitude {rm.ev_mag} earthquake occurred in the region.\n"
        f"No intensity-by-city information is available to plot for this earthquake.\n"
        f"For more details visit {rm.ev_url}"
    )
    print(report_message)

In [ ]:
print(f"Loading event {": ".join(rm.evlist[5])}")